In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

**Short answer**

The agentic loop repeatedly sends the full conversation history to the model, checks whether the model’s latest response contains any function‑call messages, runs those calls and feeds the results back into the history.  
If the latest round‑trip had **no** function calls, the loop terminates – that is the model’s “final answer” turn.

---

### How the loop works, step by step

| Step | What happens | Why it keeps the loop running |
|------|--------------|--------------------------------|
| 1. **Initial messages** | The history starts with a `developer` message that sets the role and behaviour, and the user’s question. | Sets the context for the first model turn. |
| 2. **Send to the model** | `openai_client.responses.create(...)` is called with the current `messages` list and the tool(s). | The model now decides what to do next. |
| 3. **Receive output** | The response contains an array of output items – either `message` items (plain text from the assistant) or `func

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor


In [ ]:

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xb2a96cd2d1a090906607e6b228bd73b3",
        "span_id": "0x8d806a7eee599c8a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T17:58:31.040765Z",
    "end_time": "2026-07-20T17:58:31.040813Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0d4dfdc4-cba6-4614-985e-266b32b4dee9",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [3]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            with tracer.start_as_current_span("search") as span1:
                search_results = self.search(query)
                span1.set_attribute("search_results", "search_results")
            prompt = self.build_prompt(query, search_results)
            with tracer.start_as_current_span("llm") as span2:
                answer = self.llm(prompt)
                span2.set_attribute("answer", "answer")
                span.set_attribute("input_tokens", answer.usage.input_tokens)
                span.set_attribute("output_tokens", answer.usage.output_tokens)
            span.set_attribute("rag", "rag")
            return answer.output_text

In [4]:
from starter import index, client

In [5]:
rag_traced = RAGTraced(index, client)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
# print(answer)

NameError: name 'tracer' is not defined

In [13]:
answer.output_text


'**In short:**  \nThe agentic loop is a simple “ask → run → ask again” pattern.  \nIt keeps sending the current conversation history to the model, looks at what the model returns, and if the model has asked the agent to call a tool it runs that tool, feeds the tool’s output back into the conversation, and then calls the model again. The loop stops as soon as a model reply contains **no function calls** – i.e., the model has decided it is ready to answer.\n\n---\n\n### How it works in the code\n\n```python\nwhile True:                     # ← repeat until the model is done\n    has_function_calls = False\n\n    # 1. Ask the model (the current conversation + any tool outputs)\n    response = openai_client.responses.create(\n        model="gpt‑5.4‑mini",\n        input=messages,\n        tools=[search_tool]            # the list of tools we can use\n    )\n\n    # 2. Add the model’s own reply to the history\n    messages.extend(response.output)\n\n    # 3. Walk through what the model retu

In [7]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [8]:
provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [9]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
# print(answer)

In [11]:
conn = sqlite3.connect("traces.db")
cur = conn.cursor()
cur.execute(""" select name from spans """)
names = cur.fetchall()
conn.close()
print(names)

[('search',), ('llm',), ('rag',)]


In [17]:
conn = sqlite3.connect("traces.db")
cur = conn.cursor()
# cur.execute(""" select (strftime('%s', end_time) - strftime('%s', start_time)) AS diff_seconds from spans """)
cur.execute(""" select name, (end_time - start_time) from spans """)
names = cur.fetchall()
conn.close()
print(names)

[('search', 5027456), ('llm', 31175191641), ('rag', 31391378622)]


In [18]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

In [19]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

In [20]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

In [22]:
conn = sqlite3.connect("traces.db")
cur = conn.cursor()
cur.execute(""" select input_tokens from spans where name = 'rag' """)
names = cur.fetchall()
conn.close()
print(names)

[(7174,), (7174,), (7174,), (7174,)]
